# Outliar Treatment

In [2]:
import pandas as pd 
import numpy as np

df = pd.read_csv("./data/04_Outlar_Detection.csv")

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57130 entries, 0 to 57129
Columns: 136 entries, district_name to KMeans_Outlier
dtypes: bool(93), float64(23), int64(1), object(19)
memory usage: 23.8+ MB


# data

In [6]:
# Separate outlier flags from original dataset features

flag_cols = df.select_dtypes(include="bool").columns.tolist()

feature_cols = [col for col in df.columns if col not in flag_cols]

print("Total columns:", len(df.columns))
print("Outlier flag columns:", len(flag_cols))
print("Original feature columns:", len(feature_cols))

print("\nAll original features:")
for col in feature_cols:
    print(f"{col:40} {df[col].dtype}")

print("\nAll outlier flags:")
for col in flag_cols:
    print(col)

print("\nOriginal feature shape:", df[feature_cols].shape)
print("Outlier flag shape:", df[flag_cols].shape)

Total columns: 136
Outlier flag columns: 93
Original feature columns: 43

All original features:
district_name                            object
block_tehsil_name                        object
village_name                             object
latitude                                 float64
longitude                                float64
s_no_of_scheme                           int64
scheme_type                              object
ownership                                object
total_owners_holding                     float64
social_status                            object
gender                                   object
commissioning_year                       object
scheme_depth                             float64
scheme_diameter                          float64
nearest_scheme_distance                  float64
construction_cost                        float64
machinery_cost                           float64
maintenance_cost                         float64
construction_subsidy           

### Capping / Winsorization

In [8]:
# Features excluded from outlier treatment
excluded_features = [
    col for col in df.columns
    if "ipc" in col.lower() or "ipu" in col.lower()
]

excluded_features += [
    "machinery_cost",
    "maintenance_cost",
    "construction_subsidy",
    "machinery_subsidy",
    "culturable_command_area",
    "construction_cost",
    "s_no_of_scheme",
    "latitude",
    "longitude"
]

excluded_features = list(set(excluded_features))

# Identify eligible numerical features
eligible_features = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in excluded_features
]

print("Eligible features for Capping / Winsorization:")
for col in eligible_features:
    print(col)

print("\nTotal eligible features:", len(eligible_features))

Eligible features for Capping / Winsorization:
total_owners_holding
scheme_depth
scheme_diameter
nearest_scheme_distance
horse_power_of_lifting_device

Total eligible features: 5


### IQR-Based Capping

In [11]:
# Identify features eligible for IQR-based capping

capping_summary = []

for col in eligible_features:
    flag_col = f"{col}_IQR_Outlier"

    if flag_col not in df.columns:
        continue

    outlier_count = df[flag_col].sum()

    if outlier_count > 0:
        capping_summary.append({
            "Feature": col,
            "Outlier Count": outlier_count,
            "Outlier Percentage": round(
                (outlier_count / len(df)) * 100, 2
            ),
            "Can Be Capped": "Yes"
        })
    else:
        capping_summary.append({
            "Feature": col,
            "Outlier Count": 0,
            "Outlier Percentage": 0.0,
            "Can Be Capped": "No"
        })

capping_summary = pd.DataFrame(capping_summary)
display(capping_summary)

,Feature,Outlier Count,Outlier Percentage,Can Be Capped
0,total_owners_holding,851,1.49,Yes
1,scheme_depth,2223,3.89,Yes
2,scheme_diameter,1142,2.00,Yes
3,nearest_scheme_distance,868,1.52,Yes
4,horse_power_of_lifting_device,935,1.64,Yes


### Percentile-Based Capping

In [12]:
# Identify features eligible for Percentile-Based Capping

percentile_summary = []

for col in eligible_features:
    lower_bound = df[col].quantile(0.01)
    upper_bound = df[col].quantile(0.99)

    lower_count = (df[col] < lower_bound).sum()
    upper_count = (df[col] > upper_bound).sum()

    percentile_summary.append({
        "Feature": col,
        "Lower Bound (1%)": lower_bound,
        "Upper Bound (99%)": upper_bound,
        "Lower Outliers": lower_count,
        "Upper Outliers": upper_count,
        "Total Outliers": lower_count + upper_count,
        "Can Be Capped": "Yes" if lower_count + upper_count > 0 else "No"
    })

percentile_summary = pd.DataFrame(percentile_summary)
display(percentile_summary)

,Feature,Lower Bound (1%),Upper Bound (99%),Lower Outliers,Upper Outliers,Total Outliers,Can Be Capped
0,total_owners_holding,0.00,5.000000,0,0,0,No
1,scheme_depth,7.80,60.000000,526,0,526,Yes
2,scheme_diameter,0.07,231.955914,120,0,120,Yes
3,nearest_scheme_distance,1.00,600.000000,128,0,128,Yes
4,horse_power_of_lifting_device,0.00,9.000000,0,0,0,No


### Log Transformation

In [13]:
# Identify features suitable for Log Transformation

log_summary = []

for col in eligible_features:
    skewness = df[col].skew()
    min_value = df[col].min()

    log_summary.append({
        "Feature": col,
        "Skewness": round(skewness, 3),
        "Minimum Value": min_value,
        "Right Skewed": "Yes" if skewness > 1 else "No",
        "Suitable for Log Transformation": (
            "Yes" if min_value >= 0 and skewness > 1 else "No"
        )
    })

log_summary = pd.DataFrame(log_summary)
display(log_summary)

,Feature,Skewness,Minimum Value,Right Skewed,Suitable for Log Transformation
0,total_owners_holding,1.393,0.00,Yes,Yes
1,scheme_depth,1.071,0.03,Yes,Yes
2,scheme_diameter,0.168,0.01,No,No
3,nearest_scheme_distance,0.943,0.01,No,No
4,horse_power_of_lifting_device,0.283,0.00,No,No


### Median-Based Replacement

In [14]:
# Identify features suitable for Median-Based Replacement

median_summary = []

for col in eligible_features:
    flag_col = f"{col}_IQR_Outlier"

    if flag_col not in df.columns:
        continue

    outlier_count = df[flag_col].sum()
    median_value = df.loc[~df[flag_col], col].median()

    median_summary.append({
        "Feature": col,
        "Outlier Count": outlier_count,
        "Median (Non-Outlier Values)": median_value,
        "Can Be Replaced": "Yes" if outlier_count > 0 else "No"
    })

median_summary = pd.DataFrame(median_summary)
display(median_summary)

,Feature,Outlier Count,Median (Non-Outlier Values),Can Be Replaced
0,total_owners_holding,851,1.0,Yes
1,scheme_depth,2223,24.0,Yes
2,scheme_diameter,1142,80.0,Yes
3,nearest_scheme_distance,868,120.0,Yes
4,horse_power_of_lifting_device,935,3.0,Yes


### Group-Wise Replacement

In [15]:
# Identify features suitable for Group-Wise Replacement

group_columns = [
    "district_name",
    "block_tehsil_name",
    "scheme_type",
    "ownership",
    "scheme_status"
]

group_summary = []

for col in eligible_features:
    flag_col = f"{col}_IQR_Outlier"

    if flag_col not in df.columns:
        continue

    outlier_count = df[flag_col].sum()

    if outlier_count > 0:
        for group_col in group_columns:
            if group_col in df.columns:
                group_summary.append({
                    "Feature": col,
                    "Outlier Count": outlier_count,
                    "Grouping Column": group_col,
                    "Number of Groups": df[group_col].nunique(),
                    "Can Be Replaced": "Yes"
                })

group_summary = pd.DataFrame(group_summary)
display(group_summary)

,Feature,Outlier Count,Grouping Column,Number of Groups,Can Be Replaced
0,total_owners_holding,851,district_name,34,Yes
1,total_owners_holding,851,block_tehsil_name,227,Yes
2,total_owners_holding,851,scheme_type,2,Yes
3,total_owners_holding,851,ownership,6,Yes
4,total_owners_holding,851,scheme_status,3,Yes
5,scheme_depth,2223,district_name,34,Yes
6,scheme_depth,2223,block_tehsil_name,227,Yes
7,scheme_depth,2223,scheme_type,2,Yes
8,scheme_depth,2223,ownership,6,Yes
9,scheme_depth,2223,scheme_status,3,Yes


### Row Removal

In [16]:
# Identify rows flagged by multiple outlier detection methods

outlier_flag_cols = df.select_dtypes(include="bool").columns.tolist()

df["Total_Outlier_Flags"] = df[outlier_flag_cols].sum(axis=1)

row_removal_summary = pd.DataFrame({
    "Category": [
        "Rows with no outlier flags",
        "Rows with 1 outlier flag",
        "Rows with 2 or more outlier flags",
        "Rows with 3 or more outlier flags"
    ],
    "Row Count": [
        (df["Total_Outlier_Flags"] == 0).sum(),
        (df["Total_Outlier_Flags"] == 1).sum(),
        (df["Total_Outlier_Flags"] >= 2).sum(),
        (df["Total_Outlier_Flags"] >= 3).sum()
    ]
})

display(row_removal_summary)

,Category,Row Count
0,Rows with no outlier flags,17838
1,Rows with 1 outlier flag,7593
2,Rows with 2 or more outlier flags,31699
3,Rows with 3 or more outlier flags,21352


# Save Data

In [20]:
# Remove Outlier Detection Flag Columns

flag_cols = df.select_dtypes(include="bool").columns.tolist()

df.drop(columns=flag_cols, inplace=True)
df.drop(columns=['Total_Outlier_Flags'], inplace=True)

print("Removed outlier flag columns:", len(flag_cols))
print("Remaining dataset shape:", df.shape)
display(df.head())

Removed outlier flag columns: 0
Remaining dataset shape: (57130, 43)


,district_name,block_tehsil_name,village_name,latitude,longitude,s_no_of_scheme,scheme_type,ownership,total_owners_holding,social_status,...,ipu_kharif_season,ipu_rabi_season,ipu_perennial_crop,ipu_other_season,ipu_total,under_utilised_selection,drinking_water_purpose,captured_in_previous_census,well_type,finance_source
0,Goalpara,Kuchdhowa,Siluk Pt-Ii,25.956992,90.799726,1,Tube well,Govt. Owned,0.0,Not applicable,...,4.0,1.0,0.00,0.00,5.00,Unknown,No,No,Tubewell: Medium Tube well,Government fund
1,Kokrajhar,Kokrajhar,Bhadiaguri,26.355332,90.233585,12,Tube well,Govt. Owned,0.0,Not applicable,...,0.4,0.2,0.09,0.08,0.77,Unknown,No,No,Tubewell: Shallow Tube well,Government fund
2,Goalpara,Kuchdhowa,Patiarpara Pt-I,25.971319,90.917621,1,Tube well,Govt. Owned,0.0,Not applicable,...,4.0,0.0,0.00,0.00,4.00,Unknown,No,Yes,Tubewell: Medium Tube well,Government fund
3,Dhubri,Nayeralga,Boalkamri Pt.Iii,26.210017,90.160672,4,Tube well,Owned by individual farmer,0.3,Others,...,2.0,1.5,0.00,0.00,3.50,No,No,No,Tubewell: Shallow Tube well,Own savings
4,Dhubri,Nayeralga,Boalkamri Pt.Iii,26.208178,90.161543,7,Tube well,Owned by individual farmer,1.0,Others,...,2.5,2.0,0.00,0.00,4.50,No,No,No,Tubewell: Shallow Tube well,Own savings


In [21]:
df.to_csv("./data/04_Outlar_Treatement.csv", index=False)